In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.utils.data import Dataset, Subset
from transformers import AutoTokenizer
import numpy as np
import h5py
import os
import math
import evaluate

# ==================================================================================
# 1. CONFIGURATION
# ==================================================================================

H5_FILE_PATH = "/home/poorna/data/eeg_dataset_1400_multilabel.h5"
MODEL_WEIGHTS_PATH = "best_eeg_model.pt"
TOKENIZER_PATH = "bert-base-uncased"

ENC_HIDDEN = 256
DEC_HIDDEN = 256
EMB_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.0 

NUM_CHANNELS = 62
NUM_COLORS = 9
NUM_OBJECTS = 6

COLOR_NAMES = ["Black", "Blue", "Brown", "Green", "Grey", "Orange", "Red", "White", "Yellow"]
OBJECT_NAMES = ["Animal", "Building", "Food", "Nature", "Person", "Vehicle"]

device = torch.device("cpu")
print(f"Running inference on: {device}")

# ==================================================================================
# 2. DATA LOADERS
# ==================================================================================

class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype(np.float32))
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype(np.float32))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype(np.int64))
        return eeg, meta, text

def create_stratified_split(total_samples, group_size=5):
    num_groups = total_samples // group_size
    test_indices = []
    for group_idx in range(num_groups):
        start_idx = group_idx * group_size
        group_indices = list(range(start_idx, start_idx + group_size))
        test_indices.append(group_indices[4])
    if total_samples % group_size == 5:
        start_idx = num_groups * group_size
        test_indices.append(start_idx + 4)
    return test_indices

# ==================================================================================
# 3. GRAPH
# ==================================================================================

def get_1010_geometric_graph(k_neighbors=6):
    coords_raw = [
        (0, -18, 0.51), (1, 0, 0.51), (2, 18, 0.51), (3, -23, 0.41), 
        (4, 23, 0.41), (5, -54, 0.51), (6, -49, 0.42), (7, -39, 0.33), 
        (8, -22, 0.28), (9, 0, 0.26), (10, 22, 0.28), (11, 39, 0.33), 
        (12, 49, 0.42), (13, 54, 0.51), (14, -72, 0.51), (15, -69, 0.39), 
        (16, -62, 0.28), (17, -45, 0.18), (18, 0, 0.13), (19, 45, 0.18), 
        (20, 62, 0.28), (21, 69, 0.39), (22, 72, 0.51), (23, -90, 0.51), 
        (24, -90, 0.38), (25, -90, 0.26), (26, -90, 0.13), (27, 90, 0.0), 
        (28, 90, 0.13), (29, 90, 0.26), (30, 90, 0.38), (31, 90, 0.51), 
        (32, -105, 0.51), (33, -111, 0.39), (34, -118, 0.28), (35, -135, 0.18), 
        (36, 180, 0.13), (37, 135, 0.18), (38, 118, 0.28), (39, 111, 0.39), 
        (40, 105, 0.51), (41, -120, 0.51), (42, -131, 0.42), (43, -141, 0.33), 
        (44, -158, 0.28), (45, 180, 0.26), (46, 158, 0.28), (47, 141, 0.33), 
        (48, 131, 0.42), (49, 120, 0.51), (50, -135, 0.51), (51, -147, 0.47), 
        (52, -157, 0.41), (53, 180, 0.38), (54, 157, 0.41), (55, 147, 0.47), 
        (56, 135, 0.51), (57, -150, 0.51), (58, -165, 0.51), (59, 180, 0.51), 
        (60, 165, 0.51), (61, 150, 0.51)
    ]
    num_nodes = 62
    pos = np.zeros((num_nodes, 2))
    for idx, theta, radius in coords_raw:
        rad = np.deg2rad(theta)
        pos[idx] = [radius * np.sin(rad), radius * np.cos(rad)]

    dist_matrix = np.zeros((num_nodes, num_nodes))
    for i in range(num_nodes):
        for j in range(num_nodes):
            dist_matrix[i, j] = np.linalg.norm(pos[i] - pos[j])

    edge_list = []
    for i in range(num_nodes):
        nearest = np.argsort(dist_matrix[i])[:k_neighbors + 1]
        for neighbor in nearest:
            if i != neighbor: edge_list.append([i, neighbor])
                
    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    edge_attr = torch.ones(edge_index.shape[1], dtype=torch.float)
    return edge_index.to(device), edge_attr.to(device)

# ==================================================================================
# 4. CORRECTED MODEL DEFINITIONS (MATCHING TRAINING)
# ==================================================================================

class SpatioTemporalEEGEncoder(nn.Module):
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = GCNConv(num_channels, enc_hidden)
        self.gcn2 = GCNConv(enc_hidden, enc_hidden)
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg, edge_index, edge_attr):
        batch_size = eeg.shape[0]
        num_timesteps = eeg.shape[2]
        batch_edge_index = edge_index.repeat(1, batch_size) + (torch.arange(batch_size, device=eeg.device) * self.num_channels).repeat_interleave(edge_index.shape[1])
        batch_edge_attr = edge_attr.repeat(batch_size)
        
        x = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)
        x = F.relu(self.gcn1(x, batch_edge_index, batch_edge_attr))
        x = F.relu(self.gcn2(self.dropout(x), batch_edge_index, batch_edge_attr))
        
        enc_out, enc_hid = self.rnn(x.reshape(batch_size, num_timesteps, -1))
        return enc_out.permute(1, 0, 2), enc_hid

# FIX: Restored LuongAttention class
class LuongAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim):
        super().__init__()
        self.attn = nn.Linear(enc_dim, dec_dim)

    def forward(self, decoder_hidden, encoder_outputs):
        scores = torch.bmm(decoder_hidden.permute(1, 0, 2), self.attn(encoder_outputs).permute(1, 2, 0))
        attn_weights = F.softmax(scores, dim=2)
        context = torch.bmm(attn_weights, encoder_outputs.permute(1, 0, 2))
        return context, attn_weights.squeeze(1)

# FIX: Restored variable names (color_processor, object_processor)
class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_objects, color_feature_dim=32, object_feature_dim=32):
        super().__init__()
        self.color_processor = nn.Sequential(
            nn.Linear(num_colors, 64), nn.ReLU(), nn.Linear(64, color_feature_dim)
        )
        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 64), nn.ReLU(), nn.Linear(64, object_feature_dim)
        )
        self.output_dim = color_feature_dim + object_feature_dim

    def forward(self, metadata):
        c = self.color_processor(metadata[:, :NUM_COLORS].float())
        o = self.object_processor(metadata[:, NUM_COLORS:].float())
        return torch.cat([c, o], dim=1)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, meta_features_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.dec_hidden = dec_hidden
        self.num_layers = num_layers
        enc_dim = enc_hidden * 2

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        
        # FIX: Restored use of LuongAttention class
        self.attention = LuongAttention(enc_dim, dec_hidden)

        self.rnn_input_dim = emb_dim + enc_dim + enc_dim 
        self.rnn = nn.GRU(self.rnn_input_dim, dec_hidden, num_layers, dropout=dropout if num_layers > 1 else 0)
        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.bridge = nn.Linear(enc_dim, dec_hidden)
        self.init_projector = nn.Linear(dec_hidden + meta_features_dim, dec_hidden)

    def init_hidden(self, encoder_hidden, meta_features):
        hidden = encoder_hidden.view(self.num_layers, 2, encoder_hidden.size(1), -1)
        last = torch.cat((hidden[-1][0], hidden[-1][1]), dim=1)
        bridged = torch.tanh(self.bridge(last))
        combined = torch.cat([bridged, meta_features], dim=1)
        return torch.tanh(self.init_projector(combined)).unsqueeze(0).repeat(self.num_layers, 1, 1)

    def forward(self, token, decoder_hidden, encoder_outputs, global_eeg_context):
        token = token.unsqueeze(0)
        embedded = self.dropout(self.embedding(token))
        
        # FIX: Using self.attention object
        context, attn_weights = self.attention(decoder_hidden[-1].unsqueeze(0), encoder_outputs)
        
        context_permuted = context.permute(1, 0, 2)
        global_eeg_context_unsqueezed = global_eeg_context.unsqueeze(0)
        rnn_input = torch.cat((embedded, context_permuted, global_eeg_context_unsqueezed), dim=2)
        output, hidden = self.rnn(rnn_input, decoder_hidden)
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden, context.squeeze(1)

class Seq2Seq(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_objects, enc_hidden=256, dec_hidden=256,
                 pad_id=0, dropout=0.2, emb_dim=256, dec_layers=2):
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(NUM_CHANNELS, enc_hidden, dec_layers, dropout)
        self.meta_encoder = MetadataEncoder(num_colors, num_objects)
        
        meta_dim = self.meta_encoder.output_dim
        enc_dim = enc_hidden * 2

        self.decoder = Decoder(text_vocab_size, emb_dim, enc_hidden, dec_hidden,
                               meta_dim, dec_layers, pad_id, dropout)

        # FIX: Restored full meta_head structure (Linear -> ReLU -> Norm -> Drop -> Linear)
        self.meta_head = nn.Sequential(
            nn.Linear(enc_dim, 256), 
            nn.ReLU(), 
            nn.LayerNorm(256), 
            nn.Dropout(0.3),
            nn.Linear(256, num_colors + num_objects)
        )
        self.num_colors = num_colors
        self.num_objects = num_objects

# ==================================================================================
# 5. INFERENCE LOGIC
# ==================================================================================

def decode_greedy(model, dec_hid, enc_out, global_ctx, tokenizer, max_len=30):
    curr = torch.tensor([tokenizer.cls_token_id], device=device)
    ids = []
    for _ in range(max_len):
        logits, dec_hid, _ = model.decoder(curr, dec_hid, enc_out, global_ctx)
        nxt = logits.argmax(dim=1).item()
        if nxt == tokenizer.sep_token_id: break
        ids.append(nxt)
        curr = torch.tensor([nxt], device=device)
    return tokenizer.decode(ids, skip_special_tokens=True)

def decode_beam(model, dec_hid, enc_out, global_ctx, tokenizer, beam_width=5, max_len=30):
    candidates = [(0.0, [], dec_hid)]
    completed = []
    
    for _ in range(max_len):
        next_cands = []
        for score, seq, hid in candidates:
            if len(completed) >= beam_width: break
            
            curr_token = seq[-1] if seq else tokenizer.cls_token_id
            logits, new_hid, _ = model.decoder(torch.tensor([curr_token], device=device), hid, enc_out, global_ctx)
            log_probs = F.log_softmax(logits, dim=1).squeeze(0)
            
            topk_lp, topk_id = torch.topk(log_probs, beam_width)
            for k in range(beam_width):
                idx = topk_id[k].item()
                if idx == tokenizer.sep_token_id:
                    completed.append((score - topk_lp[k].item(), seq))
                else:
                    next_cands.append((score - topk_lp[k].item(), seq + [idx], new_hid))
        
        if not next_cands: break
        candidates = sorted(next_cands, key=lambda x: x[0])[:beam_width]

    if not completed: completed = [(c[0], c[1]) for c in candidates]
    best_seq = max(completed, key=lambda x: -x[0] / (len(x[1]) + 1e-5))[1]
    return tokenizer.decode(best_seq, skip_special_tokens=True)

def get_meta_names(tensor, names, thresh=0.3):
    idxs = (torch.sigmoid(tensor) > thresh).nonzero(as_tuple=True)[0].tolist()
    return [names[i] for i in idxs if i < len(names)]

# ==================================================================================
# 6. MAIN
# ==================================================================================

if __name__ == "__main__":
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
    model = Seq2Seq(tokenizer.vocab_size, NUM_COLORS, NUM_OBJECTS).to(device)
    
    if os.path.exists(MODEL_WEIGHTS_PATH):
        print(f"Loading weights from {MODEL_WEIGHTS_PATH}...")
        model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
        print("Weights loaded.")
    else:
        raise FileNotFoundError(f"Model file not found at {MODEL_WEIGHTS_PATH}")
    
    model.eval()
    
    print("Loading Dataset...")
    dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
    test_indices = create_stratified_split(len(dataset))
    test_ds = Subset(dataset, test_indices)
    print(f"Found {len(test_ds)} test samples.")

    bleu = evaluate.load("bleu")
    rouge = evaluate.load("rouge")
    
    refs, preds_greedy, preds_beam = [], [], []
    edge_index, edge_attr = get_1010_geometric_graph()

    print("\nRunning Inference...")
    for i in range(len(test_ds)):
        eeg, meta, txt = test_ds[i]
        eeg, meta = eeg.unsqueeze(0).to(device), meta.unsqueeze(0).to(device)
        
        # Forward Pass
        enc_out, enc_hid = model.encoder(eeg, edge_index, edge_attr)
        meta_feat = model.meta_encoder(meta)
        dec_hid = model.decoder.init_hidden(enc_hid, meta_feat)
        
        # Extract Global Context correctly from Bi-GRU hidden state
        hidden_reshaped = enc_hid.view(model.encoder.rnn.num_layers, 2, 1, -1)
        last_layer = hidden_reshaped[-1]
        global_ctx = torch.cat((last_layer[0], last_layer[1]), dim=1)
        
        gt_text = tokenizer.decode(txt.tolist(), skip_special_tokens=True)
        greedy = decode_greedy(model, dec_hid, enc_out, global_ctx, tokenizer)
        beam = decode_beam(model, dec_hid, enc_out, global_ctx, tokenizer)
        
        aux = model.meta_head(global_ctx)
        p_cols = get_meta_names(aux[0, :NUM_COLORS], COLOR_NAMES)
        p_objs = get_meta_names(aux[0, NUM_COLORS:], OBJECT_NAMES)
        gt_cols = get_meta_names(meta[0, :NUM_COLORS], COLOR_NAMES, thresh=0.5)
        gt_objs = get_meta_names(meta[0, NUM_COLORS:], OBJECT_NAMES, thresh=0.5)

        refs.append([gt_text])
        preds_greedy.append(greedy)
        preds_beam.append(beam)
        
        if i < 10:
            print(f"\n--- Sample {i+1} ---")
            print(f"GT Meta:   Colors={gt_cols} | Objects={gt_objs}")
            print(f"Pred Meta: Colors={p_cols} | Objects={p_objs}")
            print(f"GT Text:   {gt_text}")
            print(f"Greedy:    {greedy}")
            print(f"Beam:      {beam}")

    print(f"\n{'='*30}\nFINAL SCORES\n{'='*30}")
    b_res = bleu.compute(predictions=preds_greedy, references=refs)
    r_res = rouge.compute(predictions=preds_greedy, references=[r[0] for r in refs])
    print(f"GREEDY -> BLEU: {b_res['bleu']:.4f} | ROUGE-L: {r_res['rougeL']:.4f}")
    
    b_res = bleu.compute(predictions=preds_beam, references=refs)
    r_res = rouge.compute(predictions=preds_beam, references=[r[0] for r in refs])
    print(f"BEAM   -> BLEU: {b_res['bleu']:.4f} | ROUGE-L: {r_res['rougeL']:.4f}")

/home/poorna/venvs/torch/lib64/python3.11/site-packages/torch_cluster/nearest.py:3: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  import scipy.cluster


Running inference on: cpu
Loading weights from best_eeg_model.pt...
Weights loaded.
Loading Dataset...
Found 5600 test samples.

Running Inference...

--- Sample 1 ---
GT Meta:   Colors=['Blue', 'Grey', 'White'] | Objects=['Building', 'Nature']
Pred Meta: Colors=['Black', 'Blue', 'Brown', 'Green', 'Grey', 'Orange', 'Red', 'White', 'Yellow'] | Objects=['Animal', 'Building', 'Food', 'Nature', 'Person', 'Vehicle']
GT Text:   a city at night with buildings lit up
Greedy:    a person is a a a
Beam:      a group of people walking down a street

--- Sample 2 ---
GT Meta:   Colors=['Blue', 'Grey', 'White'] | Objects=['Nature']
Pred Meta: Colors=['Black', 'Blue', 'Brown', 'Green', 'Grey', 'Orange', 'Red', 'White', 'Yellow'] | Objects=['Animal', 'Building', 'Food', 'Nature', 'Person', 'Vehicle']
GT Text:   a beach with a cloudy sky and a beach
Greedy:    a person is a a in a room
Beam:      a group of people swimming in the ocean

--- Sample 3 ---
GT Meta:   Colors=['Black', 'Blue', 'White'] | O

KeyboardInterrupt: 